# Interactive Maps: folium, leafmap, and sharing

We built static maps laset class, but today we're building maps used more for exploration. Features include:

- clickable markers
- popups
- various basemaps
- two maps compared side by side behind a draggable divider. 

`folium` is a "wrapper" for the JavaScript mapping library Leaflet. 

`leafmap`, adds a higher-level API over Leaflet and several other mapping engines. Both produce the same result: a single `.html` file that opens in any browser (and doesn't need Python).

By the end of this session you will be able to:

- Build a `folium` map step by step: basemap, circle markers, null-safe popups, a joined measurement, a choropleth layer, layer control, and an exported `.html` file.
- Explain the coverage gap between a station-location file and a station-measurement file, and represent it correctly in a map's popups.
- Use `leafmap` to switch basemaps and build a side-by-side comparison.
- Export an interactive map to a standalone `.html` file.

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily
%pip install -q folium leafmap

# leafmap renders through ipyleaflet, a Jupyter widget. Colab will not display
# third-party widgets until this is switched on. Outside Colab the import fails
# harmlessly and the line is skipped.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

## Building a station map, one piece at a time

`folium.Map()` builds a Leaflet map **object** in Python, which is the foundation for everything else we're going to add

This section builds one map of the Chesapeake Bay monitoring network in seven steps, from an empty basemap to a layered, exportable file. 

Run them in order and compare each map with the one before it. 

### Step 1: the empty map

`folium.Map()` takes a center as `[latitude, longitude]` (that order, the reverse of a geometry's `(x, y)`) and a zoom level. Nothing else is required.

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"

import geopandas as gpd
import pandas as pd
import folium

stations = gpd.read_file(f"{DATA_BASE_P2}/cbw_stations.geojson")
print(stations.shape, stations.crs)

station_map = folium.Map(location=[38.5, -76.5], zoom_start=7)
station_map

**What step 1 results:** 
- an OpenStreetMap basemap centered on the Bay, with nothing on it. 

The object was built WITHOUT a network call. What does that mean? Python needs only the center and zoom, and the tile images load in the browser once the cell's output renders.

### Step 2: a basemap and markers

`tiles=` selects the basemap for the map. `folium` accepts any name from the `xyzservices` provider list - this one is a muted gray canvas with no API key required. 

- `folium.Marker()` places a pin at a latitude and longitude
- `.add_to()` attaches it to the map. The loop does this once per station.

In [ ]:
station_map = folium.Map(location=[38.5, -76.5], zoom_start=7, tiles="Esri.WorldGrayCanvas")  # NEW: tiles

for _, row in stations.iterrows():                               # NEW: iterate through the DataFrame
    folium.Marker(location=[row["latitude"], row["longitude"]]).add_to(station_map) # NEW: add marker for each station

station_map

**Step 2 results:** 
1. a quieter gray basemap (Esri's World Gray Canvas, designed to sit under data)
2. 166 blue pins piled along the Bay. 

Pins have a fixed pixel size, so at this zoom they overlap into a solid mass along the Potomac and the Eastern Shore, and clicking one does nothing (yet)

### Understanding Python's `for _, row` and `.iterrows()` Syntax

When we loop through a pandas DataFrame using `.iterrows()`, we encounter two key concepts:

1. **`stations.iterrows()`**:
   - A pandas method that loops over the rows of a DataFrame
   - In each iteration, it returns a **tuple** containing two elements: `(index, row_data)`
   - `index` is the row label (often an integer starting from 0).
   - `row_data` is a pandas Series containing all the column values for that specific row (e.g., `latitude`, `longitude`)

2. **The Underscore (`_`) Placeholder**:
   - Because `.iterrows()` always yields a pair of values, Python syntax expects us to unpack it into two variables.
   - If we don't plan to use the row's `index` in our code (and only care about the coordinates and attributes inside `row`), we assign the index to `_`.
   - In Python, `_` is a conventional, throwaway variable name indicating: *"A value is returned here, but I am intentionally ignoring it."*




### Step 3: circle markers and a popup

`folium.CircleMarker()` draws a small circle instead of a pin. It has 
- a radius in pixels
- a stroke color
- a fill

`popup=` attaches text that appears on click - a string is the simplest popup (for demo purposes)

In [ ]:
station_map = folium.Map(location=[38.5, -76.5], zoom_start=7, tiles="Esri.WorldGrayCanvas")

for _, row in stations.iterrows():
    folium.CircleMarker(                                         # NEW: CircleMarker replaces Marker
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="darkblue",
        fill=True,
        fill_opacity=0.8,
        popup=row["station_id"],                                 # NEW: add popup with station_id
    ).add_to(station_map)

station_map

**Step 3 results:** 
- 166 small dots that retain their spacing
- a station ID on click. 

The map now shows where the network is, but a station ID alone tells the reader VERY little

(the file has a `station_type` and a `waterbody` for each station thought)

### Step 4: popups with attributes

`folium.Popup()` accepts HTML, so a popup can contain several lines. The attributes have gaps: 23 of the 166 stations have a null `station_type` and the same 23 have a null `waterbody`, which `stations["station_type"].isna().sum()` confirms. 

If we build a pop-up from those columns, we also have to handle the nulls.

**Before running.** For a station whose `station_type` is null, what does the f-string `f"type: {row['station_type']}"` produce: an error, an empty string, or the word `nan`?

In [ ]:
# Test the above question:
# Pick one station with a null station_type and inspect the value itself.
row = stations[stations["station_type"].isna()].iloc[0]
missing = row["station_type"]

print("value:", repr(missing), "| type:", type(missing).__name__)
print("f-string gives:", f"type: {missing}")
print("pd.isna() gives:", pd.isna(missing))


In [ ]:
station_map = folium.Map(location=[38.5, -76.5], zoom_start=7, tiles="Esri.WorldGrayCanvas")

for _, row in stations.iterrows():
    stype = "unknown" if pd.isna(row["station_type"]) else row["station_type"]  # NEW: handle missing values
    wbody = "unknown" if pd.isna(row["waterbody"]) else row["waterbody"]  # NEW: handle missing values
    popup_text = f"<b>{row['station_id']}</b><br>type: {stype}<br>waterbody: {wbody}"  # NEW: create HTML popup text
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="darkblue",
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=220),           # NEW: HTML popup
    ).add_to(station_map)

station_map

**Step 4 results:** 
- three-line popups, with "unknown" for the 23 stations missing a type and waterbody. 
- If we hadn't checked for the unknown situation, the f-string would have printed `type: nan`

The source of the null was the GeoJSON as a floating-point `NaN`, not Python's `None`, and `NaN` counts as true, so `row["station_type"] or "unknown"` would have passed it straight through. <-- Yes, this is a bit confusing


Regardless, the test `pd.isna()` catches both conditions.

### Step 5: join the measurements

`cbw_tidal_wq_2025.csv` contains 2025 water-quality readings.

The next cell retains each station's latest Total Nitrogen (TN) reading, then joins it onto all 166 stations with a left merge, and finally  builds a popup that shows the value where one exists (if not, says so). The marker color follows the same split.

In [ ]:
wq = pd.read_csv(f"{DATA_BASE_P2}/cbw_tidal_wq_2025.csv")                       # NEW: Read the water quality data
latest_tn = (                                                                   # NEW: Get the latest TN reading for each station
    wq[wq["parameter"] == "TN"] # NEW: Filter for TN readings
    .sort_values("date") # NEW: Sort by date
    .groupby("station_id") # NEW: Group by station_id
    .last() # NEW: Get the last (latest) reading for each station
    .reset_index()[["station_id", "date", "value", "unit"]] # NEW: Keep only the relevant columns
)
stations_tn = stations.merge(latest_tn, on="station_id", how="left")            # NEW Merge the latest TN readings with the stations GeoDataFrame
print(f"{stations_tn['value'].notna().sum()} of {len(stations_tn)} stations have a 2025 TN reading in this extract")

station_map = folium.Map(location=[38.5, -76.5], zoom_start=7, tiles="Esri.WorldGrayCanvas")

for _, row in stations_tn.iterrows():                                           # NEW: loop over the merged table
    if pd.notna(row["value"]):                                                  # NEW: popup and color by coverage
        popup_text = f"<b>{row['station_id']}</b><br>Latest 2025 TN: {row['value']:.3f} {row['unit']} ({row['date']})"
        color = "green"
    else:
        popup_text = f"<b>{row['station_id']}</b><br>No 2025 TN reading in this extract"
        color = "lightgray"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=220),
    ).add_to(station_map)

station_map

**Step 5 results:** 
- **86 of 166** stations contain a TN reading
- The other 80 markers remain on the map in light gray (rather than being dropped), and each have a popup stating that no 2025 TN reading exists. 

***COMMUNICATION NOTE:*** The wording matters! The station is not closed and the network has not stopped monitoring it; this particular data extract does not cover it. 

A map that plotted only the 86 successful joins would misrepresent the other 80 stations as nonexistent.

**PRACTICE EXERCISE:** Rebuild `latest_tn` with `parameter == "WTEMP"` (water temperature) instead of `"TN"`, rejoin, and recount. The provenance notes describe `WTEMP` as mostly continuous sensor readings and `TN` as discrete laboratory samples. Is the station coverage the same 86, or different?

In [ ]:
# Try it here:

## YACM (yet another choropleth map)

The map's next layer needs a value per county.

We have `cbw_counties.geojson`, but it doesn't have great ready-made values to shade by

Perhaps a spatial join can help?

### Prep work

In [ ]:
counties = gpd.read_file(f"{DATA_BASE_P2}/cbw_counties.geojson") # Read the counties shapefile
print(counties.shape, counties.crs)

stations_matched = stations.to_crs(counties.crs) # Match the CRS of the stations to the counties for spatial join

# Perform a spatial join - note the use of the "within" predicate to find stations that are within counties
joined = gpd.sjoin(stations_matched, counties[["geoid", "name", "state", "geometry"]], how="left", predicate="within")

# Count the number of stations in each county
station_counts = joined.groupby("geoid").size().rename("station_count")

# Merge the station counts back to the counties GeoDataFrame
counties_counts = counties.merge(station_counts, on="geoid", how="left")

# Fill NaN values with 0 and convert to integer
counties_counts["station_count"] = counties_counts["station_count"].fillna(0).astype(int)

# print to test
print(f"{(counties_counts['station_count'] > 0).sum()} of {len(counties_counts)} counties have at least one station")
print(counties_counts.sort_values("station_count", ascending=False)[["name", "state", "station_count"]].head(5))

**Result:** **42 of 205** counties contain at least one tidal station; most of the watershed's counties are inland, upstream of the tidal reach. 

The District of Columbia (`geoid` `11001`) leads with **17** stations along its short tidal shoreline, followed by four Maryland counties with 8 to 13 (Prince George's 13, Dorchester 10, Charles and Queen Anne's 8 each). 

## YOUR TURN

Form groups

Worth through the rest of the notebook in the remaining time

### Step 6: a choropleth layer

`folium.Choropleth()` shades polygons by a column, 

NOTE, it takes the GeoDataFrame **TWICE**:
1. once as the geometry (`geo_data`) 
2. once as the table (`data`)

`key_on` identifies the property that links them. 

Adding it ***before*** the markers puts the counties ***underneath*** the points. 

As the watershed extends into New York, I changed the map zoom by one level.

In [ ]:
station_map = folium.Map(location=[38.5, -76.5], zoom_start=6, tiles="Esri.WorldGrayCanvas")  # NEW: zoom 7 -> 6

folium.Choropleth(                                   # NEW: the county layer goes on first, so points draw on top
    geo_data=counties_counts,                        # NEW: use the counties_counts GeoDataFrame as the geo_data
    data=counties_counts,                            # NEW: use the counties_counts GeoDataFrame as tabular data
    columns=["geoid", "station_count"],              # NEW: specify the columns to use for the choropleth
    key_on="feature.properties.geoid",               # NEW: specify the key to match the choropleth data to the geo_data
    fill_color="YlOrRd",                             # NEW: specify the fill color scheme
    fill_opacity=0.75,                               # YOUR TODO: specify the fill opacity - what is the range of valid values?
    line_opacity=0.3,                                # YOUR TODO: specify the line opacity - what is the range of valid values?
    legend_name="",                                  # YOUR TODO: specify the legend name
    name="",                                         # YOUR TODO: specify the layer name
).add_to(station_map)                                # NEW: add the choropleth to the map

# Add the monitoring stations as circle markers
for _, row in stations_tn.iterrows():
    if pd.notna(row["value"]):
        popup_text = f"<b>{row['station_id']}</b><br>Latest 2025 TN: {row['value']:.3f} {row['unit']} ({row['date']})"
        color = "green"
    else:
        popup_text = f"<b>{row['station_id']}</b><br>No 2025 TN reading in this extract"
        color = "lightgray"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=220),
    ).add_to(station_map)

station_map # show the map

**Step 6 results:**
- a shaded county layer under the station dots, with a legend. 

**NOTE:** The 163 zero-station counties take the palette's lightest color rather than gray or blank: `folium.Choropleth` treats `0` as a value, which is correct here, since a count of zero is a measurement, not a gap. 


### Step 7: layer control and export

1. `folium.FeatureGroup()` collects the markers into a named layer
2. `folium.LayerControl()` adds the widget that toggles layers on and off
3. `.save()` writes the whole map to one `.html` file that opens in any browser with no Python, no Colab runtime, no server. 



In [ ]:
station_map = folium.Map(location=[38.5, -76.5], zoom_start=6, tiles="Esri.WorldGrayCanvas")

folium.Choropleth(                                   
    geo_data=counties_counts,
    data=counties_counts,
    columns=["geoid", "station_count"],
    key_on="feature.properties.geoid",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name="Monitoring stations per county",
    name="Station density",
).add_to(station_map)

station_layer = folium.FeatureGroup(name="Stations")                            # NEW: create a feature group for the stations

for _, row in stations_tn.iterrows():
    if pd.notna(row["value"]):
        popup_text = f"<b>{row['station_id']}</b><br>Latest 2025 TN: {row['value']:.3f} {row['unit']} ({row['date']})"
        color = "green"
    else:
        popup_text = f"<b>{row['station_id']}</b><br>No 2025 TN reading in this extract"
        color = "lightgray"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=220),
    ).add_to(station_layer)                                                 # NEW: add to the group
station_layer.add_to(station_map)                               # NEW: add the group to the map

folium.LayerControl().add_to(station_map)                       # NEW: add a layer control to toggle the choropleth and station layers
station_map.save("cbw_station_map.html")                        # NEW: save the map to an HTML file
print("saved cbw_station_map.html")
station_map

**Step 7 Result:** 

the same map with a layer widget in the upper right, listing "Station density" and "Stations" as checkboxes, so the dots can be hidden to read the counties and shown again. 

### About the file we saved
- `cbw_station_map.html` is a single file of a few megabytes, since the 205 county polygons are embedded in it, written to Colab's session storage. 
- Download it from the Files pane (folder icon at the left)



## Questions to answer and tasks to complete

1. The loop runs 166 times, but the layer control lists only two layers. Why not 167?
2. Where in this cell is the station count used? Where is the TN value used? Point to the line for each.
3. Delete the LayerControl line and predict what you would see. Then run it.
4. Change the legend name to say "tidal" somewhere. Now change it again so it no longer matches what the colors show. What is the smallest edit you could make that would make this map be "dishonest"?
5. Make the gray markers smaller than the green ones. Which single line changes, and HOW does it need to change?
6. Set the choropleth's fill opacity to 1.0. What did you just lose, and who would notice?
7. Change the filename so it carries today's date, built with an f-string rather than typed. Why might that matter for Lab 6?
8. First, color the markers by `station_type` instead of by TN coverage: `"darkblue"` for `"main"` (mainstem), `"teal"` for `"trib"` (tributary), and `"lightgray"` for the 23 nulls, using `pd.isna()` as in step 4. How many of each appear? 
9. Second, change the choropleth's `fill_color` to `"BuPu"` and its `legend_name` to match. Does the darkest county change, or only its color?

In [ ]:
## Do it here:



## IF WE HAVE TIME...

### And now for something sort of different.... leafmap: basemaps

`leafmap.Map()` builds on the same Leaflet foundation as `folium` with a higher-level API. 

`.add_basemap()` replaces the base tile layer by name from a library of more than 150 built-in options, with no tile-URL template required.

### Let's build another example

In [ ]:
import leafmap

m1 = leafmap.Map(center=[38.17, -76.52], zoom=6)
m1.add_basemap("Esri.WorldImagery")
m1.add_gdf(stations, layer_name="CB stations") # Add the monitoring stations as a GeoDataFrame
m1

**Result:** the map object and its basemap and layer references build immediately. 

`leafmap` records `"Esri.WorldImagery"` as the requested tile source and `"cbw_stations"` as a vector layer;  both are fetched and rendered only when the map widget displays. 


In [ ]:
# A second worked example: a different basemap and a different layer
m2 = leafmap.Map(center=[38.17, -76.52], zoom=6)
m2.add_basemap("Esri.WorldGrayCanvas") # gray basemap for contrast with the stations
m2.add_gdf(counties, layer_name="CB counties") # counties instead of stations
m2

**Result:** the same offline-safe construction, now with `cbw_counties.geojson`'s 205 polygons as the vector layer and a muted gray canvas instead of imagery. 

**Practice exercise.** Build a third `leafmap.Map()` with `"OpenTopoMap"` as its basemap and `stations` filtered to `station_type == "trib"` as its vector layer.

In [ ]:
### Try it here:


## leafmap: split-panel comparison

`m.split_map()` allows us to use two basemaps side by side with a draggable divider. 

***WHY?***
1. It's cool
2. Useful for comparing two renderings of the same geography rather than switching one basemap back and forth from memory.

**Before running.** A split map with `"OpenStreetMap"` on the left and `"Esri.WorldImagery"` on the right, both centered on the Chesapeake Bay. What could a reader learn by dragging the divider across this pair that a single static map could not show? 

In [ ]:
m3 = leafmap.Map(center=[38.17, -76.52], zoom=7)
m3.split_map(
    left_layer="OpenStreetMap",
    right_layer="Esri.WorldImagery",
    left_label="OpenStreetMap (drawn)",
    right_label="Esri World Imagery (satellite)",
)
m3

**Result:** Cool slider stuff of somewhat questionable utility (in this form)

### We can do this for any pair of basemaps - here's 2 from ESRI

In [ ]:
# A second worked example: two basemaps from one provider, a subtler comparison
m4 = leafmap.Map(center=[38.17, -76.52], zoom=7)
m4.split_map(
    left_layer="Esri.WorldStreetMap",
    right_layer="Esri.WorldGrayCanvas",
    left_label="World Street Map (full color)",
    right_label="World Gray Canvas (muted)",
)
m4

**Practice exercise.** Build a split map of your own basemap pair, centered on Cuyahoga County (`center=[41.5, -81.6], zoom=10`). Choose two basemaps that make a useful comparison and write one sentence on why that pair rather than any two names from the list.

In [ ]:
### Try it here:

## Exporting a leafmap map

`leafmap`'s `Map` supports the same standalone export as `folium`: `.to_html()` writes a self-contained file that doesn't need the Colab runtime (or any runtime) to view.

In [ ]:
m1.to_html("cbw_stations_imagery_map.html")
print("saved cbw_stations_imagery_map.html")

**Result:** a standalone `.html` file in the working directory, openable in any browser

## Static and interactive maps

- When might we want to use static maps? Interactive?

- Which tool is better for which task?

## Wrap-up and next steps

**Practice** (no submission):

1. Rebuild the TN-popup map with every marker colored by `station_type` (`"main"`, `"trib"`, or missing) instead of by whether it has a 2025 reading. Is the main-versus-tributary split geographically clustered or mixed?
2. On the choropleth map, change `predicate="within"` to `predicate="intersects"` in the spatial join and recompute `station_counts`. Does any county's count change?
3. Build a `leafmap.Map()` centered on a place you know well, with `"Esri.WorldImagery"` as the basemap and no data layer.

**This week's reading:** ["Getting started"](https://python-visualization.github.io/folium/latest/getting_started.html) (Folium documentation) and ["Choropleth Mapping"](https://geographicdata.science/book/notebooks/05_choropleth.html) (*Geographic Data Science with Python*, chapter 5).

**Next week:** spatial weights, formalizing "neighbor" with `libpysal`, in preparation for Moran's I and LISA cluster maps.

*Adapted from Qiusheng Wu, "Introduction to GIS Programming" (geog-312), CC BY 4.0.*